We will find rrs matchups on copernicus marine products for the hplc dataset processed in notebook '1_read_and_preprocess_hplc_world_data'

In [1]:
import pandas as pd
import xarray as xr
import numpy as np
import datetime
import re
import copernicusmarine as cm
import pickle

from src.data.matchups import match_up
from pathlib import Path
from difflib import get_close_matches
from tqdm import tqdm

p_pro = Path('../../../data/processed/hplc_world/')

pigment_names = ['chlide_a[mg*m^3]', 'chla[mg*m^3]', 'chlb[mg*m^3]', 'chlc1+c2[mg*m^3]',
       'fucox[mg*m^3]', "19'hxfcx[mg*m^3]", "19'btfcx[mg*m^3]",
       'diadino[mg*m^3]', 'allox[mg*m^3]', 'diatox[mg*m^3]', 'zeaxan[mg*m^3]',
       'beta_car[mg*m^3]', 'peridinin[mg*m^3]']


# pigments_threshold = [0.00248, 0.05878, 0.003  , 0.00518, 0.003  , 0.01302, 0.0036 , 0.00968, 0.001  , 0.0018 , 0.00844, 0.00242, 0.001]

input_vars_legacy_6 = ['412', '442', '490', '510', '560', '673']
input_vars_legacy_5 = ['412', '442', '490', '560', '673']
input_vars_OLCI_13 = ["400", "412", "442", "490", "510", "560", "620", "665", "673", "681", "708", "778", "865"]
input_vars_OLCI_11 = ["400", "412", "442", "490", "510", "560", "620", "665", "673", "681", "708"]
input_vars_sat_OLCI_11 = ["RRS400", "RRS412", "RRS443", "RRS490", "RRS510", "RRS560", "RRS620", "RRS665", "RRS674", "RRS681", "RRS709"]
input_vars_sat_OLCI_13 = ["RRS400", "RRS412_5", "RRS442_5", "RRS490", "RRS510", "RRS560", "RRS620", "RRS665", "RRS673_75", "RRS681_25", "RRS708_75", "RRS778_75", "RRS865"]


dict_wv_6 = {'RRS412': '412', 'RRS443': '442', 'RRS490': '490', 'RRS510': '510', 'RRS555': '560', 'RRS670': '673'}
dict_wv_5 = {'RRS412': '412', 'RRS443': '442', 'RRS490': '490', 'RRS555': '560', 'RRS670': '673'}
dict_wv_OLCI_11 = dict(zip(input_vars_sat_OLCI_11, input_vars_OLCI_11))
dict_wv_OLCI_13 = dict(zip(input_vars_sat_OLCI_13, input_vars_OLCI_13))

LAT, LON, TIME = 'lat', 'lon', 'time'

# Thresholds for mathcup
TIME_TH = pd.Timedelta(days=1)
LAT_TH = 0.1
LON_TH = 0.1

# Matchup region extension (window)
TIME_WINDOW = pd.Timedelta(days=1)
LAT_WINDOW = 0.06
LON_WINDOW = 0.06

#### Load hplc data

In [2]:
hplc = xr.load_dataset(p_pro / 'hplc.nc')
# hplc_2 = xr.load_dataset('../../../data/processed/dataset_hplc_multi/y.nc')

In [3]:
match_id, match_lat, match_lon, match_time = hplc['Id'].values, hplc['lat'].values, hplc['lon'].values, hplc['time'].values

#### Lazy load of Copernicus datasets

In [5]:
id_multi_med = 'cmems_obs-oc_med_bgc-reflectance_my_l3-multi-1km_P1D'
id_OLCI_med = 'cmems_obs-oc_med_bgc-reflectance_my_l3-olci-300m_P1D'
id_multi = 'cmems_obs-oc_glo_bgc-reflectance_my_l3-multi-4km_P1D'
# id_OLCI_300 = 'cmems_obs-oc_glo_bgc-reflectance_my_l3-olci-300m_P1D'
id_OLCI_4 = 'cmems_obs-oc_glo_bgc-reflectance_my_l3-olci-4km_P1D'
id_multi_bs = 'cmems_obs-oc_blk_bgc-reflectance_my_l3-multi-1km_P1D'
id_OLCI_bs = 'cmems_obs-oc_blk_bgc-reflectance_my_l3-olci-300m_P1D'
rrs_multi_med = cm.open_dataset(dataset_id = id_multi_med)
rrs_OLCI_med = cm.open_dataset(dataset_id = id_OLCI_med)
rrs_multi_bs = cm.open_dataset(dataset_id = id_multi_bs)
rrs_OLCI_bs = cm.open_dataset(dataset_id = id_OLCI_bs)
rrs_multi = cm.open_dataset(dataset_id = id_multi)
rrs_OLCI_4 = cm.open_dataset(dataset_id = id_OLCI_4)

# rrs_OLCI_300 = cm.open_dataset(dataset_id = id_OLCI_300)



INFO - 2026-01-21T16:13:40Z - Selected dataset version: "202311"
INFO - 2026-01-21T16:13:40Z - Selected dataset part: "default"
INFO - 2026-01-21T16:13:45Z - Selected dataset version: "202211"
INFO - 2026-01-21T16:13:45Z - Selected dataset part: "default"
INFO - 2026-01-21T16:13:51Z - Selected dataset version: "202311"
INFO - 2026-01-21T16:13:51Z - Selected dataset part: "default"
INFO - 2026-01-21T16:13:57Z - Selected dataset version: "202211"
INFO - 2026-01-21T16:13:57Z - Selected dataset part: "default"
INFO - 2026-01-21T16:14:02Z - Selected dataset version: "202311"
INFO - 2026-01-21T16:14:02Z - Selected dataset part: "default"
INFO - 2026-01-21T16:14:07Z - Selected dataset version: "202207"
INFO - 2026-01-21T16:14:07Z - Selected dataset part: "default"


In [45]:
# rrs_world = cm.open_dataset(dataset_id = 'cmems_obs-oc_glo_bgc-reflectance_my_l3-multi-4km_P1D')
# rrs_med  = cm.open_dataset(dataset_id = 'cmems_obs-oc_med_bgc-reflectance_my_l3-multi-1km_P1D')
# rrs_med_olci  = cm.open_dataset(dataset_id = 'cmems_obs-oc_med_bgc-reflectance_my_l3-olci-300m_P1D')

In [6]:
# rrs_world = rrs_world.rename({'latitude': LAT, 'longitude': LON}).sortby(LAT).sortby(LON)

rrs_multi_med = rrs_multi_med.rename({'latitude': LAT, 'longitude': LON}).sortby(LAT).sortby(LON)
rrs_OLCI_med = rrs_OLCI_med.rename({'latitude': LAT, 'longitude': LON}).sortby(LAT).sortby(LON)

rrs_multi_bs = rrs_multi_bs.rename({'latitude': LAT, 'longitude': LON}).sortby(LAT).sortby(LON)
rrs_OLCI_bs = rrs_OLCI_bs.rename({'latitude': LAT, 'longitude': LON}).sortby(LAT).sortby(LON)

rrs_multi = rrs_multi.rename({'latitude': LAT, 'longitude': LON}).sortby(LAT).sortby(LON)
rrs_OLCI_4 = rrs_OLCI_4.rename({'latitude': LAT, 'longitude': LON}).sortby(LAT).sortby(LON)


#### Def main function to perform matchups

In [7]:
match_ups_multi_med_list = match_up(match_id, match_lat, match_lon, match_time, rrs_multi_med)


  0%|          | 0/53894 [00:00<?, ?it/s]

In [8]:
match_ups_OLCI_med_list = match_up(match_id, match_lat, match_lon, match_time, rrs_OLCI_med)


  0%|          | 0/53894 [00:00<?, ?it/s]

In [9]:
match_ups_multi_bs_list = match_up(match_id, match_lat, match_lon, match_time, rrs_multi_bs)

  0%|          | 0/53894 [00:00<?, ?it/s]

In [10]:
match_ups_OLCI_bs_list = match_up(match_id, match_lat, match_lon, match_time, rrs_OLCI_bs)


  0%|          | 0/53894 [00:00<?, ?it/s]

In [11]:
match_ups_multi_list = match_up(match_id, match_lat, match_lon, match_time, rrs_multi)


  0%|          | 0/53894 [00:00<?, ?it/s]

In [12]:
match_ups_OLCI_list = match_up(match_id, match_lat, match_lon, match_time, rrs_OLCI_4)


  0%|          | 0/53894 [00:00<?, ?it/s]

In [13]:
# match_ups_med_olci_list = match_up(match_id, match_lat, match_lon, match_time, rrs_med_olci)
print(len(match_ups_multi_med_list), len(match_ups_OLCI_med_list), len(match_ups_multi_bs_list), len(match_ups_OLCI_bs_list), len(match_ups_multi_list), len(match_ups_OLCI_list))

1927 0 0 0 32348 1284


In [14]:
# Eliminate matchups when time window is not the spcecified
time_length = TIME_WINDOW.days * 2 + 1

match_ups_multi_med_time = [ m for m in match_ups_multi_med_list if len(m.time[0]) == time_length]
match_ups_multi_med_time_lon = [ m for m in match_ups_multi_med_time if len(m.lon[0]) == 9]

# match_ups_OLCI_med_time = [ m for m in match_ups_OLCI_med_list if len(m.time[0]) == time_length]
# match_ups_OLCI_med_time_lon = [ m for m in match_ups_OLCI_med_time if len(m.lon[0]) == 35]


# match_ups_multi_bs_time = [ m for m in match_ups_multi_bs_list if len(m.time[0]) == time_length]
# match_ups_multi_bs_time_lon = [ m for m in match_ups_multi_bs_time if len(m.lon[0]) == 9]

# match_ups_OLCI_bs_time = [ m for m in match_ups_OLCI_bs_list if len(m.time[0]) == time_length]
# match_ups_OLCI_bs_time_lon = [ m for m in match_ups_OLCI_bs_time if len(m.lon[0]) == 35]

match_ups_multi_time = [ m for m in match_ups_multi_list if len(m.time[0]) == time_length]
match_ups_multi_time_lon = [ m for m in match_ups_multi_time if len(m.lon[0]) == 3]

match_ups_OLCI_time = [ m for m in match_ups_OLCI_list if len(m.time[0]) == time_length]
match_ups_OLCI_time_lon = [ m for m in match_ups_OLCI_time if len(m.lon[0]) == 3]

print(len(match_ups_multi_med_time_lon), len(match_ups_multi_time_lon), len(match_ups_OLCI_time_lon))

1927 32300 1284


In [21]:
xar_multi_matchups = xr.concat(match_ups_multi_time_lon,  dim='Id')
xar_OLCI_matchups = xr.concat(match_ups_OLCI_time_lon,  dim='Id')

In [22]:
# xar_multi_bs_matchups = xr.concat(match_ups_multi_bs_time_lon,  dim='Id')
# xar_OLCI_bs_matchups = xr.concat(match_ups_OLCI_bs_time_lon,  dim='Id')

In [23]:
xar_multi_med_matchups = xr.concat(match_ups_multi_med_time_lon,  dim='Id')
# xar_OLCI_med_matchups = xr.concat(match_ups_OLCI_med_time_lon,  dim='Id')

### Change variable names to the ones of the models trained


In [24]:
xar_multi_med_matchups = xar_multi_med_matchups.rename(dict_wv_6)[input_vars_legacy_6]
# xar_OLCI_med_matchups = xar_OLCI_med_matchups.rename(dict_wv_OLCI_13)[input_vars_OLCI_13]

In [25]:
# xar_multi_bs_matchups = xar_multi_bs_matchups.rename(dict_wv_6)[input_vars_legacy_6]
# xar_OLCI_bs_matchups = xar_OLCI_bs_matchups.rename(dict_wv_OLCI_13)[input_vars_OLCI_13]

In [26]:
xar_multi_matchups = xar_multi_matchups.rename(dict_wv_5)[input_vars_legacy_5]
xar_OLCI_matchups = xar_OLCI_matchups.rename(dict_wv_OLCI_11)[input_vars_OLCI_11]

In [27]:
xar_multi_med_matchups_loaded = xar_multi_med_matchups.load()
xar_multi_med_matchups_loaded.to_netcdf(p_pro / 'matchups_rrs_multi_med.nc')

In [28]:
# xar_OLCI_med_matchups_loaded = xar_OLCI_med_matchups.load()
# xar_OLCI_med_matchups_loaded.to_netcdf(p_pro / 'matchups_rrs_OLCI_med.nc')

In [29]:
# xar_multi_bs_matchups_loaded = xar_multi_bs_matchups.load()
# xar_multi_bs_matchups_loaded.to_netcdf(p_pro / 'matchups_rrs_multi_bs.nc')

In [30]:
# xar_OLCI_bs_matchups_loaded = xar_OLCI_bs_matchups.load()
# xar_OLCI_bs_matchups_loaded.to_netcdf(p_pro / 'matchups_rrs_OLCI_bs.nc')

In [31]:
xar_multi_matchups_loaded = xar_multi_matchups.load()
xar_multi_matchups_loaded.to_netcdf(p_pro / 'matchups_rrs_multi.nc')

In [32]:
xar_OLCI_matchups_loaded = xar_OLCI_matchups.load()
xar_OLCI_matchups_loaded.to_netcdf(p_pro / 'matchups_rrs_OLCI.nc')